# WildlifeVision YOLO11 Trainer

Train a bird detector from the reviewed dataset exported by WildlifeVision, then export the ONNX model and class labels for the browser app.

**Workflow:** import and label images in WildlifeVision, prepare and verify the bounding boxes, save the YOLO dataset ZIP, then open this notebook with **Train in Colab**.

In [ ]:
!git clone https://github.com/EdebN/curly-system.git
%cd /content/curly-system
!pip install -q -r training/requirements.txt

## 1. Select the T4 runtime

Choose **Runtime → Change runtime type → T4 GPU** and save before training.

## 2. Upload your dataset and (optionally) your previous checkpoint

Upload the `WildlifeVision-YOLO-dataset.zip` exported from Bird Library. To continue improving an existing detector, select its `best.pt` file in the same upload dialog. If you do not upload a checkpoint, training starts from `yolo11n.pt`.

**Important:** ONNX files are inference exports and cannot be fine-tuned. Keep the `.pt` checkpoint. For new species, include the previous species' labelled images in the new dataset as well, so the detector does not simply forget them.

In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()
dataset_files = [name for name in uploaded if name.lower().endswith('.zip')]
checkpoint_files = [name for name in uploaded if name.lower().endswith('.pt')]
if len(dataset_files) != 1:
    raise ValueError(f'Upload exactly one dataset ZIP. Found: {dataset_files}')
if len(checkpoint_files) > 1:
    raise ValueError(f'Upload at most one .pt checkpoint. Found: {checkpoint_files}')
dataset_zip = str(Path(dataset_files[0]).resolve())
model_checkpoint = str(Path(checkpoint_files[0]).resolve()) if checkpoint_files else 'yolo11n.pt'
print('Dataset:', dataset_zip)
print('Starting checkpoint:', model_checkpoint)
print('Using previous trained weights.' if checkpoint_files else 'Starting from the standard YOLO11n checkpoint.')

## 3. Train YOLO11 and export ONNX

Run this cell to display the training button, then click it. If you supplied `best.pt`, the trainer starts from those weights. With the same species list, it fine-tunes the detector on the expanded dataset. If you added species, Ultralytics transfers compatible weights and adapts the detection head to the new class list.

In [ ]:
import subprocess, sys, torch, ipywidgets as widgets
from IPython.display import display

train_button = widgets.Button(description='Train YOLO11', button_style='success', icon='play', tooltip='Fine-tune and export the detector')
train_output = widgets.Output(layout={'border': '1px solid #888', 'max_height': '420px', 'overflow_y': 'auto'})

def start_yolo_training(_):
    train_button.disabled = True
    train_button.description = 'Training…'
    with train_output:
        print('Preparing YOLO11 training…')
        device = '0' if torch.cuda.is_available() else 'cpu'
        print('Training device:', device)
        command = [
            sys.executable, 'training/train_yolo.py', dataset_zip,
            '--model', model_checkpoint,
            '--epochs', '100', '--imgsz', '640', '--batch', '-1',
            '--workers', '2', '--device', device,
            '--output', '/content/wildlifevision-training',
            '--name', 'bird-detector'
        ]
        try:
            process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
            for line in process.stdout:
                print(line, end='')
            exit_code = process.wait()
            if exit_code != 0:
                raise subprocess.CalledProcessError(exit_code, command)
            print('\nTraining and ONNX export completed. Download the new ONNX, labels, and checkpoint below.')
            train_button.description = 'Training complete'
            train_button.button_style = 'success'
        except Exception as error:
            print('\nTraining failed:', error)
            train_button.description = 'Retry training'
            train_button.button_style = 'danger'
        finally:
            train_button.disabled = False

train_button.on_click(start_yolo_training)
display(train_button, train_output)

## 4. Download the updated detector

Download all three files. Use the new `best.onnx` and `best.labels.json` in WildlifeVision. Keep the new `best.pt` as the checkpoint for the next training round.

In [ ]:
from google.colab import files
from pathlib import Path

weights = Path('/content/wildlifevision-training/runs/bird-detector/weights')
for filename in ['best.onnx', 'best.labels.json', 'best.pt']:
    path = weights / filename
    if not path.is_file():
        raise FileNotFoundError(f'Expected training output is missing: {path}')
    print('Downloading', filename)
    files.download(str(path))

## 5. Load the updated model in WildlifeVision

In **Live camera**, choose the new `best.onnx`, upload the matching `best.labels.json`, then select **Load YOLO11**. The labels file must always match the ONNX model.

For the next update, repeat the process with the latest `best.pt` and a dataset ZIP containing all species you want the detector to retain.